In [3]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
# 1. 설정 및 클래스 정의
LABEL_DIR = 'dataset_50k/Training/labels'
CLASS_MAP = {
    0: 'eye_opened', 
    1: 'eye_closed', 
    2: 'mouth_opened', 
    3: 'mouth_closed', 
    4: 'face'
}

import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

def _parse_label_file(file_path):
    """개별 파일을 처리하는 헬퍼 함수 (병렬 처리를 위해 분리)"""
    class_ids = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_ids.append(int(parts[0]))
    except Exception as e:
        return []
    return class_ids

def analyze_label_distribution_parallel(label_path, mapping, max_workers=None):
    label_files = glob.glob(os.path.join(label_path, '*.txt'))
    
    if not label_files:
        print(f"오류: {label_path} 경로에 라벨 파일이 없습니다.")
        return None

    all_class_ids = []
    
    # 병렬 처리 시작
    print(f"--- 데이터 분석 시작 (총 {len(label_files)} 파일) ---")
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # 진행률 표시를 위한 tqdm 설정
        futures = {executor.submit(_parse_label_file, f): f for f in label_files}
        
        for future in tqdm(as_completed(futures), total=len(label_files), desc="Processing Labels"):
            all_class_ids.extend(future.result())

    # ID를 클래스 이름으로 변환
    class_names = [mapping.get(cid, 'Unknown') for cid in all_class_ids]
    
    # 데이터프레임 변환
    df = pd.DataFrame(class_names, columns=['class_name'])
    
    # 분석 결과 출력
    stats = df['class_name'].value_counts()
    print("\n--- 클래스별 분포 통계 ---")
    print(stats)
    
    # 시각화
    plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")
    sns.countplot(data=df, x='class_name', order=list(mapping.values()), palette='viridis')
    plt.title('Label Distribution Analysis (Parallel)', fontsize=15)
    plt.xlabel('Class Name', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    return df

# 실행
dist_df = analyze_label_distribution_parallel(LABEL_DIR, CLASS_MAP)

--- 데이터 분석 시작 (총 49995 파일) ---


Processing Labels:   0%|          | 0/49995 [00:08<?, ?it/s]


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [7]:
dist_df.to_csv('label_df.csv', index=False)